# PA2: 01_sft_sanity

In [ ]:
import base64, io, zipfile, os, subprocess
from pathlib import Path
root = Path('/kaggle/working/pa2')
root.mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(root)
os.chdir(root)
subprocess.run(['python', '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch, transformers, datasets
print({'torch': torch.__version__, 'transformers': transformers.__version__, 'datasets': datasets.__version__})
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, 'Select GPU T4 x2'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2)), 'Expected two T4 GPUs'
subprocess.run(['python', '-m', 'pytest', '-q', 'tests'], check=True)


In [ ]:
import json
candidates = list(Path('/kaggle/input').rglob('sft/best/config.json'))
assert len(candidates) == 1, f'Expected one SFT best checkpoint, found {candidates}'
sft = candidates[0].parent
data_source = sft.parents[1] / 'data'
prompts = data_source/'alpaca_eval_300_seed42.json'
assert prompts.is_file()
for model, name in [('Qwen/Qwen2.5-0.5B', 'qwen2.5-0.5b-base'), (str(sft), 'sft')]:
    target = f'outputs/sanity/{name}.json'
    subprocess.run(['python','-m','scripts.generate','--model',model,'--prompts',str(prompts),
                    '--output',target,'--generator',name,'--limit','3',
                    '--repetition-penalty','1.0','--no-repeat-ngram-size','0'],check=True)
    print(json.dumps(json.loads(Path(target).read_text()),indent=2))
